# 🛡️ GigShield: Local Model Training Pipeline
Run this notebook locally or on Kaggle/Colab after generating your hybrid datasets. This notebook processes the data, trains the models using Scikit-Learn and XGBoost, and generates the `*.pkl` weights.

In [ ]:
# 1. Import necessary libraries
!pip install pandas xgboost scikit-learn joblib networkx

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.model_selection import train_test_split
import joblib

### 📊 Step 1: Load and Prepare Hybrid Data

In [ ]:
# Load the dataset you generated via `dataset_merger.py` (ensure it exists in your directory)
# df = pd.read_csv('processed_hybrid_training_data.csv')

# For notebook execution purposes without the real CSV yet, generating a mock dataframe:
num_samples = 5000
df_mock = pd.DataFrame({
    'inter_order_variance': np.random.uniform(0.1, 5.0, num_samples),
    'rider_tenure_days': np.random.randint(1, 365, num_samples),
    'claims_count': np.random.randint(0, 10, num_samples),
    'trust_score': np.random.randint(20, 100, num_samples),
    'zone_risk_variance': np.random.uniform(1.0, 5.0, num_samples),
    'time_of_day': np.random.randint(0, 23, num_samples),
    'precip_last_2_hours': np.random.uniform(0, 50, num_samples), # mm of rain
    'avg_velocity_kmph': np.random.uniform(10, 80, num_samples),  # GPS speeds
    # Target 1: Weekly Premium Cost generated loosely off variance/claims
    'target_loss_cost': np.random.uniform(10, 50, num_samples), 
    # Target 2: Income Loss calculated loosely off time and rain
    'target_income_loss': np.random.uniform(0, 600, num_samples) 
})
print("Data Loaded! Shape:", df_mock.shape)

### 🤖 Step 2: Train Model 1 -> Dynamic Premium Prediction (XGBoost)
Trains on rider loyalty patterns, tenure, and zone risk to determine base weekly premium.

In [ ]:
# Define Features for Premium Model
features_premium = ['inter_order_variance', 'rider_tenure_days', 'claims_count', 'trust_score', 'zone_risk_variance']
X_prem = df_mock[features_premium]
y_prem = df_mock['target_loss_cost']

X_train, X_test, y_train, y_test = train_test_split(X_prem, y_prem, test_size=0.2, random_state=42)

premium_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4)
premium_model.fit(X_train, y_train)

print("Premium Model XGBoost Trained! Score (R^2):", premium_model.score(X_test, y_test))

joblib.dump(premium_model, 'premium_model.pkl')
print("✅ Saved: premium_model.pkl")

### 💰 Step 3: Train Model 2 -> Income Estimation Model (Random Forest)
Estimates real earnings lost per hour factoring in demand drops (rush hours) and trailing weather effects.

In [ ]:
# Define Features for Income Model
features_income = ['time_of_day', 'precip_last_2_hours', 'zone_risk_variance']
X_inc = df_mock[features_income]
y_inc = df_mock['target_income_loss']

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_inc, y_inc, test_size=0.2, random_state=42)

income_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
income_model.fit(X_train_i, y_train_i)

print("Income Model RF Trained! Score (R^2):", income_model.score(X_test_i, y_test_i))

joblib.dump(income_model, 'income_model.pkl')
print("✅ Saved: income_model.pkl")

### 🛑 Step 4: Train Model 3 -> Structural Fraud Detection (Isolation Forest)
Detects anomalous GPS patterns like spoofing/teleportation using unsupervised learning based on trajectory vectors.

In [ ]:
# Inject 3% obvious anomalies into velocity speeds to simulate GPS spoofing software (e.g., 200 km/h in dense city)
df_mock.loc[df_mock.sample(frac=0.03).index, 'avg_velocity_kmph'] = np.random.uniform(150, 400, int(num_samples*0.03))

features_fraud = ['avg_velocity_kmph', 'inter_order_variance']
X_fraud = df_mock[features_fraud]

# Train Isolation Forest (contamination is the expected % of anomalies)
fraud_model = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)
fraud_model.fit(X_fraud)

# Verify it caught anomalies (-1 is anomaly, 1 is normal)
predictions = fraud_model.predict(X_fraud)
print("Fraud Validation Check | Anomalies Detected:", list(predictions).count(-1))

joblib.dump(fraud_model, 'fraud_model.pkl')
print("✅ Saved: fraud_model.pkl")

## 🚀 Task Complete!
The output of this notebook is three files:
1. `premium_model.pkl`
2. `income_model.pkl`
3. `fraud_model.pkl`

**Please download these three `.pkl` files and place them in the `gigshield_final/` project folder containing `app.py`.** 
Once placed, GigShield can be configured to load them dynamically.